# LOGIT-02 — Modèle ajusté, applicable à toute la population

Même méthode que LOGIT-01, une seule variable change — et c'est celle qui rend le
modèle utilisable.

## Ce qui change

**La cohérence SIREN vient de notre chaîne.** Le modèle fidèle utilise la
cohérence calculée avec le SIREN corrigé par la revue manuelle, renseignée sur
9 079 candidats seulement. Ici, on utilise `coherence_EGE_ANS`, produite par notre
propre sirenisation et disponible sur toute la population.

C'est exactement la condition de production : en exploitation, le SIREN corrigé
n'existe pas. Le coefficient sera plus faible, et c'est normal — il est honnête.

**L'accord entre jeux de données est conservé.** Il est désormais disponible sur
l'ensemble des structures, ce qui n'était pas le cas avant réception du fichier
de propositions.

## Ce que ça permet

Estimer sur les mêmes structures examinées, puis **appliquer à la population
entière**. C'est l'objet de la seconde partie.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))

import numpy as np
import pandas as pd

from src import logit
from src.drees import (
    charger_propositions, propositions_ans, charger_echantillon,
    table_apprentissage, COL_COHERENCE_ANS,
)
from src.display import afficher_tableau
from config.settings import DREES_DIR, LOGIT_DIR

LOGIT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 60)

PROPOSITIONS = DREES_DIR / "ege_siret_proposals.parquet"
ECHANTILLON = DREES_DIR / "ege_review_sample.xlsx"
SEUIL = 0.90

## 1. Estimation

In [2]:
print("Propositions :")
propositions = charger_propositions(PROPOSITIONS)

print("\nEchantillon de revue :")
echantillon = charger_echantillon(ECHANTILLON)

print("\nTable d'apprentissage :")
table = table_apprentissage(propositions, echantillon, "ege")

y = table["y"]
poids = table["calibrated_weight"]
groupes = table["EJ"]

Propositions :
   141,924 candidats, 102,926 structures
   origine : {'both': 63133, 'none': 35248, 'ANS': 29863, 'Adrien': 13680}

Echantillon de revue :
   916 structures examinees, 839 avec un identifiant retenu

Table d'apprentissage :
   92,996 structures avec une proposition de notre chaine
   698 structures examinees avec une proposition
   11 a fermer ecartees -> 687 exploitables
   proposition correcte : 441 / 687 (85.4 % pondere)


### Les variables

`coherence_EGE_ANS` prend trois valeurs : cohérent, incohérent, ou « EGE seul »
quand la personne morale n'a pas été validée et qu'aucune comparaison n'est
possible. La modalité de référence est « incohérent ».

In [3]:
def construire_variables(d):
    coherence = d[COL_COHERENCE_ANS].fillna("")
    return pd.DataFrame({
        "SIRET initial = oui":
            d["is_initial_siret"].fillna(False).astype(bool).astype(int),
        "Statut ANS = VALIDE_FORT":
            (d["statut_EGE_ANS"].fillna("") == "VALIDE_FORT").astype(int),
        "Accord total":
            (d["datasets_consistency_EGE"] == "Total consistency").astype(int),
        "Accord partiel":
            (d["datasets_consistency_EGE"] == "Partial consistency").astype(int),
        "SIREN coherent = oui":
            (coherence == "Cohérent").astype(int),
        "SIREN sans comparaison":
            coherence.isin(["EGE seul", ""]).astype(int),
    }, index=d.index)

X = construire_variables(table)
print(X.sum().rename("structures concernees").to_string())

SIRET initial = oui         277
Statut ANS = VALIDE_FORT    167
Accord total                242
Accord partiel               38
SIREN coherent = oui        365
SIREN sans comparaison       35


In [4]:
modele = logit.ajuster(X, y, poids)
afficher_tableau(logit.odds_ratios(modele), "Coefficients et odds ratios",
                 max_lignes=8)

Variable,Coefficient,Odds ratio,IC bas,IC haut,p-value
const,-0.885000,0.410000,0.390000,0.430000,0.000000
SIRET initial = oui,1.658000,5.250000,4.870000,5.660000,0.000000
Statut ANS = VALIDE_FORT,1.338000,3.810000,3.480000,4.170000,0.000000
Accord total,1.429000,4.180000,3.850000,4.530000,0.000000
Accord partiel,-3.188000,0.040000,0.040000,0.050000,0.000000
SIREN coherent = oui,2.306000,10.040000,9.420000,10.700000,0.000000
SIREN sans comparaison,1.249000,3.490000,2.880000,4.220000,0.000000


## 2. Performance

À comparer avec LOGIT-01. L'écart mesure ce que valait réellement la cohérence
corrigée par la revue.

In [5]:
proba = logit.predire_hors_echantillon(X, y, poids, groupes, n_plis=5)

afficher_tableau(logit.metriques(y, proba, poids), "Metriques hors echantillon",
                 max_lignes=8)
afficher_tableau(logit.calibration(y, proba, poids), "Calibration", max_lignes=8)

Metrique,Valeur
Score de Brier,0.067800
Brier de reference (taux moyen),0.124700
Reduction du Brier,45.6 %
AUC,0.907300
Prediction moyenne,0.847300
Taux observe,0.854000
Observations,687


Classe,Probabilite min,Probabilite max,Prediction moyenne,Taux observe,Effectif
1,0.003000,0.313000,0.151000,0.282000,161
2,0.340000,0.799000,0.646000,0.588000,114
3,0.810000,0.941000,0.894000,0.868000,139
4,0.944000,0.978000,0.963000,0.984000,143
5,0.981000,1.000000,0.992000,0.984000,130


In [6]:
seuils = logit.table_seuils(y, proba, poids)
afficher_tableau(seuils, "Volume et precision selon le seuil", max_lignes=12)

ligne = seuils[seuils["Seuil"] == SEUIL].iloc[0]
print(f"\nAu seuil de {SEUIL:.2f} : {ligne['Acceptes (%)']:.1f} % acceptes, "
      f"{ligne['Corrects parmi acceptes (%)']:.1f} % corrects")

Seuil,Acceptes (%),Corrects parmi acceptes (%),Erreur residuelle (%),Corrects parmi rejetes (%),Structures acceptees
0.500000,86.000000,94.900000,5.100000,26.800000,492
0.600000,85.900000,94.900000,5.100000,27.200000,488
0.700000,85.100000,95.300000,4.700000,29.000000,458
0.800000,79.400000,96.800000,3.200000,41.200000,412
0.850000,76.400000,98.400000,1.600000,43.400000,353
0.900000,75.700000,98.500000,1.500000,44.700000,327
0.920000,74.500000,98.400000,1.600000,47.400000,318
0.950000,64.300000,98.300000,1.700000,62.100000,227
0.970000,58.300000,98.500000,1.500000,67.100000,182



Au seuil de 0.90 : 75.7 % acceptes, 98.5 % corrects


---

## 3. Application à toute la population

Le modèle est appliqué aux structures pour lesquelles notre chaîne propose un
identifiant. Les probabilités produites ici ne sont pas vérifiables : il n'y a pas
d'annotations hors échantillon.

Ce qu'on observe, c'est leur **distribution**. Une concentration inattendue sur un
statut ou une catégorie serait un signal à creuser avant toute généralisation.

In [7]:
population = propositions_ans(propositions)
X_population = construire_variables(population)

population = population.assign(
    probabilite=logit.appliquer(modele, X_population).round(4))
population["accepte"] = population["probabilite"] >= SEUIL

print(f"{len(population):,} structures scorees")
print(f"   acceptees au seuil {SEUIL:.2f} : {int(population['accepte'].sum()):,} "
      f"({population['accepte'].mean() * 100:.1f} %)")

   92,996 structures avec une proposition de notre chaine
92,996 structures scorees
   acceptees au seuil 0.90 : 73,634 (79.2 %)


### Distribution des probabilités

In [8]:
tranches = pd.cut(population["probabilite"],
                  [0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0],
                  labels=["< 0,50", "0,50-0,70", "0,70-0,80",
                          "0,80-0,90", "0,90-0,95", "> 0,95"])
distribution = tranches.value_counts().sort_index().rename("Structures").reset_index()
distribution.columns = ["Tranche", "Structures"]
distribution["Part (%)"] = (distribution["Structures"] / len(population) * 100).round(1)
afficher_tableau(distribution, "Distribution des probabilites", max_lignes=8)

Tranche,Structures,Part (%)
"< 0,50",12138,13.100000
"0,50-0,70",1503,1.600000
"0,70-0,80",0,0.000000
"0,80-0,90",5721,6.200000
"0,90-0,95",6286,6.800000
"> 0,95",67348,72.400000


### Croisements

Trois lectures. Par statut du pipeline : le modèle doit suivre le score sans s'y
réduire. Par cohérence SIREN : c'est la variable la plus discriminante. Par
catégorie d'établissement : elle n'entre pas dans le modèle, un écart marqué
signalerait une population mal couverte par l'échantillon d'estimation.

In [9]:
for colonne, titre in [("statut_EGE_ANS", "Statut du pipeline"),
                       (COL_COHERENCE_ANS, "Coherence SIREN"),
                       ("datasets_consistency_EGE", "Accord entre jeux de donnees")]:
    resume = population.groupby(population[colonne].fillna("(vide)")).agg(
        Structures=("probabilite", "size"),
        Probabilite_moyenne=("probabilite", "mean"),
        Acceptees=("accepte", "sum"))
    resume["Part acceptee (%)"] = (resume["Acceptees"] / resume["Structures"] * 100).round(1)
    resume["Probabilite_moyenne"] = resume["Probabilite_moyenne"].round(3)
    afficher_tableau(resume.reset_index(names=titre), titre, max_lignes=10)

Statut du pipeline,Structures,Probabilite_moyenne,Acceptees,Part acceptee (%)
VALIDE,61670,0.821000,43380,70.300000
VALIDE_FORT,31326,0.975000,30254,96.600000


Coherence SIREN,Structures,Probabilite_moyenne,Acceptees,Part acceptee (%)
Cohérent,78069,0.951000,70788,90.700000
EGE seul,1241,0.812000,487,39.200000
Incohérent,13686,0.431000,2359,17.200000


Accord entre jeux de donnees,Structures,Probabilite_moyenne,Acceptees,Part acceptee (%)
Partial consistency,2388,0.194000,0,0.000000
Total consistency,60745,0.985000,60361,99.400000
Total inconsistency,29863,0.698000,13273,44.400000


In [10]:
categories = population.groupby("EGE__libcategetab").agg(
    Structures=("probabilite", "size"),
    Probabilite_moyenne=("probabilite", "mean"),
    Part_acceptee=("accepte", "mean"))
categories["Probabilite_moyenne"] = categories["Probabilite_moyenne"].round(3)
categories["Part_acceptee"] = (categories["Part_acceptee"] * 100).round(1)
categories = categories[categories["Structures"] >= 100].sort_values("Probabilite_moyenne")

afficher_tableau(categories.head(15).reset_index(), "Categories les moins bien scorees",
                 max_lignes=15)
afficher_tableau(categories.tail(10).reset_index(), "Categories les mieux scorees",
                 max_lignes=10)

EGE__libcategetab,Structures,Probabilite_moyenne,Part_acceptee
Centre gratuit d'information de dépistage et de diagnostic,166,0.539000,22.300000
Centre de soins et de prévention,101,0.624000,49.500000
Etablissement Expérimental pour personnes handicapées,104,0.653000,32.700000
Protection Maternelle et Infantile (P.M.I.),715,0.662000,48.400000
Maisons Relais - Pensions de Famille,873,0.664000,46.800000
Lieux de soins non programmés,311,0.677000,52.700000
Ctre.Accueil/ Accomp.Réduc.Risq.Usag. Drogues (C.A.A.R.U.D.),156,0.708000,46.200000
Centre de santé sexuelle,398,0.710000,51.300000
Centres Locaux Information Coordination P.A .(C.L.I.C.),271,0.710000,48.300000
Equipe Mobile Médico-Sociale Précarité,101,0.711000,43.600000


EGE__libcategetab,Structures,Probabilite_moyenne,Part_acceptee
Maison de Santé pour Maladies Mentales,298,0.920000,88.900000
Dispositif d'appui à la coordination,135,0.925000,90.400000
Etablissement et Service d'Aide par le Travail (E.S.A.T.),1323,0.927000,88.900000
Structure Dispensatrice à domicile d'Oxygène à usage médical,811,0.932000,89.400000
Services de Prévention et de Santé au Travail (SPST),415,0.934000,87.200000
Etablissement d'hébergement pour personnes âgées dépendantes,6871,0.939000,91.200000
"Centre hospitalier, ex Hôpital local",191,0.943000,92.700000
Etablissement de Soins Chirurgicaux,106,0.954000,94.300000
Etablissement de Soins Pluridisciplinaire,441,0.956000,94.800000
Pharmacie d'Officine,19700,0.968000,93.600000


### Divergences entre le modèle et le pipeline

Deux situations symétriques : une proposition que le score n'a pas validée mais
que le modèle accepte, et l'inverse. Ce sont les cas à examiner en priorité.

In [11]:
VALIDES = ["VALIDE", "VALIDE_FORT"]

acceptees_non_validees = population[
    (~population["statut_EGE_ANS"].isin(VALIDES)) & population["accepte"]]
validees_non_acceptees = population[
    population["statut_EGE_ANS"].isin(VALIDES) & (~population["accepte"])]

print(f"non validees par le score, acceptees par le modele : {len(acceptees_non_validees):,}")
print(f"validees par le score, refusees par le modele       : {len(validees_non_acceptees):,}")

cols = ["EGE", "EGE__rs", "statut_EGE_ANS", COL_COHERENCE_ANS,
        "datasets_consistency_EGE", "siret_proposal", "probabilite"]
if len(acceptees_non_validees):
    afficher_tableau(acceptees_non_validees[cols].sort_values("probabilite", ascending=False),
                     "Acceptees par le modele, non validees par le score", max_lignes=12)
if len(validees_non_acceptees):
    afficher_tableau(validees_non_acceptees[cols].sort_values("probabilite"),
                     "Validees par le score, refusees par le modele", max_lignes=12)

non validees par le score, acceptees par le modele : 0
validees par le score, refusees par le modele       : 19,362


EGE,EGE__rs,statut_EGE_ANS,coherence_EGE_ANS,datasets_consistency_EGE,siret_proposal,probabilite
860784420,EANM ANDRE RIDEAU,VALIDE,Incohérent,Partial consistency,30053625700102,0.016700
350055000,SAAD ADMR RETIERS,VALIDE,Incohérent,Partial consistency,77775168600059,0.016700
420015422,SPASAD ADMR,VALIDE,Incohérent,Partial consistency,78982361400027,0.016700
860015015,SERVICE DE SOINS SNCF,VALIDE,Incohérent,Partial consistency,95407682400778,0.016700
420012304,SAMSAH ADENE MEDICO SOCIAL ST-ETIENNE,VALIDE,Incohérent,Partial consistency,51976774300023,0.016700
060787769,CMP CAGNES SUR MER,VALIDE,Incohérent,Partial consistency,17060431800106,0.016700
400782603,SAAD ADMR MONTFORT EN CHALOSSE,VALIDE,Incohérent,Partial consistency,49508274500011,0.016700
410006191,EHPAD RESIDENCE SAVIGNY (ORPEA),VALIDE,Incohérent,Partial consistency,40125156602333,0.016700
860785856,CMP MONTMORILLON,VALIDE,Incohérent,Partial consistency,20005535800028,0.016700
890971104,SAAD ADMR CHAILLEY,VALIDE,Incohérent,Partial consistency,77868363100023,0.016700


## 4. Export

In [12]:
FICHIER = LOGIT_DIR / "EGE_logit_modele_ajuste.xlsx"

cols_sortie = ["EGE", "EJ", "EGE__rs", "EGE__libcategetab", "statut_EGE_ANS",
               COL_COHERENCE_ANS, "datasets_consistency_EGE", "is_initial_siret",
               "siret_proposal", "probabilite", "accepte"]
cols_sortie = [c for c in cols_sortie if c in population.columns]

with pd.ExcelWriter(FICHIER, engine="openpyxl") as writer:
    logit.odds_ratios(modele).to_excel(writer, sheet_name="Coefficients", index=False)
    logit.metriques(y, proba, poids).to_excel(writer, sheet_name="Metriques", index=False)
    logit.calibration(y, proba, poids).to_excel(writer, sheet_name="Calibration", index=False)
    seuils.to_excel(writer, sheet_name="Seuils", index=False)
    distribution.to_excel(writer, sheet_name="Distribution", index=False)
    categories.reset_index().to_excel(writer, sheet_name="Par_categorie", index=False)
    population[cols_sortie].to_excel(writer, sheet_name="Predictions", index=False)

print(f"Export : {FICHIER}")
print(f"   {len(population):,} structures scorees")

Export : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/model_logit/EGE_logit_modele_ajuste.xlsx
   92,996 structures scorees
